# <span style="color:#ff8c00">00 · Data preparation</span>

Download, extract, and build the datasets, weights, split tables, and decoded-image cache for the VQA benchmark.

### Disk budget

| Stage | Size | Contents |
|---|---|---|
| **Zips** (`paths.zip_dir`) | ~22 GB | Raw question, annotation, COCO image, and embedding archives |
| **Extracted** (`paths.raw_dir`) | ~29 GB | Unpacked JSON questions/annotations, COCO images, word vectors (gensim format), and BERT models |
| **Image-encoder weights** (`raw_dir/models/cnn`, `models/torchvision`) | ~16 GB | timm and torchvision weights, downloaded by notebook 03 |
| **Processed** (`paths.processed_dir`) | Up to ~114 GB | Split parquet tables, 70 GB decoded-image cache (448×448), and BERT question states (19 GB base + 26 GB large, built on first BERT run) |
| **Temporary features** | Variable | Cached pooled vectors and 7×7 feature grids during encoder evaluation |
| **Total recommended** | **~200 GB** | Full storage budget for reproducing all benchmark runs |

# <span style="color:#ff8c00">Setup</span>

In [ ]:
from vqa_lab.config import load_paths

paths = load_paths()
print("Repository root:", paths.root)
print("Zip directory:   ", paths.zip_dir)
print("Raw directory:   ", paths.raw_dir)
print("Processed dir:   ", paths.processed_dir)

# <span style="color:#ff8c00">Download raw archives</span>

Download the VQA v2 questions, annotations, COCO 2014 training and validation image sets, and static word embeddings (GloVe and fastText) into `paths.zip_dir`.

Existing files are skipped. Downloads write to a temporary file and rename upon completion so an interrupted download restarts cleanly on the next run. Note: `test2015` images are not needed.

In [ ]:
from pathlib import Path
from vqa_lab.data import fetch_archive

VQA_URL = "https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/"
ARCHIVES = [  # (archive URL, destination folder under raw_dir)
    (VQA_URL + "v2_Questions_Train_mscoco.zip", "questions"),
    (VQA_URL + "v2_Questions_Val_mscoco.zip", "questions"),
    (VQA_URL + "v2_Questions_Test_mscoco.zip", "questions"),
    (VQA_URL + "v2_Annotations_Train_mscoco.zip", "annotations"),
    (VQA_URL + "v2_Annotations_Val_mscoco.zip", "annotations"),
    ("http://images.cocodataset.org/zips/train2014.zip", ""),
    ("http://images.cocodataset.org/zips/val2014.zip", ""),
]
WORD_VECTORS = [  # (archive URL, question encoder): converted once to gensim's format below, not unpacked
    ("https://nlp.stanford.edu/data/glove.840B.300d.zip", "glove"),
    ("https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip", "fasttext"),
]

for url, _ in ARCHIVES + WORD_VECTORS:
    fetch_archive(url, paths.zip_dir / Path(url).name)

# <span style="color:#ff8c00">Extract archives</span>

Extract the downloaded archives into `paths.raw_dir`:
- Question archives → `raw_dir/questions/`
- Annotation archives → `raw_dir/annotations/`
- COCO image archives → `raw_dir/` (`train2014.zip` and `val2014.zip` already contain their respective folder)

The GloVe and fastText zips are not unpacked here: the next step reads them.

Extractions are skipped if every member is already on disk with its full size.

In [ ]:
from vqa_lab.data import extract_archive

for url, sub in ARCHIVES:
    extract_archive(paths.zip_dir / Path(url).name, paths.raw_dir / sub)

# <span style="color:#ff8c00">Convert word vectors</span>

GloVe and fastText are read once with gensim and saved in its own format (`text.text_vectors_to_kv`, about 10 minutes in all); the unpacked text files are deleted again and the zips stay. Every later load memory-maps the saved vectors in about a second (`text.load_vectors`), as for word2vec.

In [ ]:
from vqa_lab.text import STATIC_FILES, text_vectors_to_kv

for url, arm in WORD_VECTORS:   # skipped once the converted file exists
    print(arm, "->", text_vectors_to_kv(paths.zip_dir / Path(url).name, paths.raw_dir / STATIC_FILES[arm]))

# <span style="color:#ff8c00">Download models and word vectors</span>

Download BERT models and word2vec from the Hugging Face Hub using `data.fetch_hub`.
- `google-bert/bert-base-uncased` → `raw_dir/models/bert-base-uncased`
- `google-bert/bert-large-uncased` → `raw_dir/models/bert-large-uncased`
- `fse/word2vec-google-news-300` → `raw_dir/embeddings/word2vec`

CNN backbone weights do not require pre-downloading: the library automatically downloads them on first use into `raw_dir/models/cnn/` (timm) and `raw_dir/models/torchvision/`.

In [ ]:
from vqa_lab.data import fetch_hub

BERT_FILES = ["config.json", "model.safetensors", "tokenizer.json", "tokenizer_config.json", "vocab.txt"]
DOWNLOADS = [  # (hub repository, folder, files that must all exist, files to fetch or None for the whole repository)
    ("google-bert/bert-base-uncased", "models/bert-base-uncased", BERT_FILES, BERT_FILES),
    ("google-bert/bert-large-uncased", "models/bert-large-uncased", BERT_FILES, BERT_FILES),
    ("fse/word2vec-google-news-300", "embeddings/word2vec",
     ["word2vec-google-news-300.model", "word2vec-google-news-300.model.vectors.npy"], None),
]
for repo, folder, needed, patterns in DOWNLOADS:
    fetch_hub(repo, paths.raw_dir / folder, needed=needed, patterns=patterns)

# <span style="color:#ff8c00">Build dataset tables</span>

Parse the raw question and annotation JSON files into unified parquet tables for each split (`train`, `val`, `test_dev`, and `test`).

If the four parquet files already exist in `paths.tables_dir`, they are read directly from disk.

In [ ]:
import pandas as pd
from vqa_lab.data import load_tables

tables = load_tables(paths)   # built from the JSON files once, then read from parquet
pd.DataFrame({s: {"questions": len(t), "images": t["image_id"].nunique()} for s, t in tables.items()}).T

# <span style="color:#ff8c00">Build decoded-image cache</span>

Pre-decode and resize all COCO 2014 training and validation images to 448×448 bicubic RGB arrays, saved as memory-mapped uint8 numpy arrays (`pixels.npy` and `ids.npy`).

This takes about 7 minutes on a modern multi-core workstation and requires ~70 GB disk space. It accelerates all downstream CNN feature extraction runs by bypassing JPEG decompression. Image IDs are strictly sorted with `np.sort` to preserve benchmark row order. The builder resumes automatically if interrupted.

In [ ]:
import numpy as np
from vqa_lab.image import build_decoded_cache, decoded_dir

for split, table in (("train2014", "train"), ("val2014", "val")):
    ids = np.sort(tables[table]["image_id"].unique())   # the row order the benchmark used
    build_decoded_cache(paths.raw_dir / split, split, ids, decoded_dir(paths, split))   # resumes, skips finished rows
    print(split, f"{len(ids):,} images ready")

# <span style="color:#ff8c00">Verify dataset preparation</span>

Check that all required raw data files, word vectors, models, parquet tables, and image caches are present.

In [ ]:
expected = [
    ("Questions (train)", paths.raw_dir / "questions/v2_OpenEnded_mscoco_train2014_questions.json"),
    ("Questions (val)", paths.raw_dir / "questions/v2_OpenEnded_mscoco_val2014_questions.json"),
    ("Questions (test)", paths.raw_dir / "questions/v2_OpenEnded_mscoco_test2015_questions.json"),
    ("Annotations (train)", paths.raw_dir / "annotations/v2_mscoco_train2014_annotations.json"),
    ("Annotations (val)", paths.raw_dir / "annotations/v2_mscoco_val2014_annotations.json"),
    ("Images (train2014)", paths.raw_dir / "train2014"),
    ("Images (val2014)", paths.raw_dir / "val2014"),
    ("GloVe vectors", paths.raw_dir / STATIC_FILES["glove"]),
    ("fastText vectors", paths.raw_dir / STATIC_FILES["fasttext"]),
    ("word2vec model", paths.raw_dir / STATIC_FILES["word2vec"]),
    ("BERT-base model", paths.raw_dir / "models/bert-base-uncased/model.safetensors"),
    ("BERT-large model", paths.raw_dir / "models/bert-large-uncased/model.safetensors"),
    ("Table: train.parquet", paths.tables_dir / "train.parquet"),
    ("Table: val.parquet", paths.tables_dir / "val.parquet"),
    ("Table: test_dev.parquet", paths.tables_dir / "test_dev.parquet"),
    ("Table: test.parquet", paths.tables_dir / "test.parquet"),
    ("Decoded cache: train2014", decoded_dir(paths, "train2014") / "pixels.npy"),
    ("Decoded cache: val2014", decoded_dir(paths, "val2014") / "pixels.npy"),
]

pd.DataFrame([{"component": name, "present": p.exists()} for name, p in expected])